In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('/content/heart.csv')
df

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='all')

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique()

In [ ]:
categorical_df = df.select_dtypes(include = 'object').columns
categorical_df

In [ ]:
UniqueInChestPainType = df['ChestPainType'].unique()
UniqueInChestPainType

In [ ]:
Resting_ECG = df['RestingECG'].unique()
Resting_ECG

In [ ]:
ExercisingAgina = df['ExerciseAngina'].unique()
ExercisingAgina

In [ ]:
ST_Slope = df['ST_Slope'].unique()
ST_Slope

Converting Categorial data to Numeric Data
 Sex : M = 0 , F = 1
 ChestPainType :
 ATA  = 0
 NAP  = 1
 ASY = 2
 TA = 3
 Resting ECG : Normal = 0
 ST = 1
 LVH = 2
 ExercisingAngina: N= 0, Y = 1
 ST_Slope: Up = 0
 Flat = 1
 Down = 2


In [ ]:
# Conversion Code [ May Faced Some Issues Later Due to Pandas]
#for col in categorical_df:
#print(col)
#print((df[col].unique()),list(range(df[col].nunique())))
#df[col].replace((df[col].unique()),range(df[col].nunique()),inplace = True)
#print('*'*90)
#print()
# SAfer Code [Same Logic]
# Conversion Code (SAFE VERSION)
for col in categorical_df:
    print(col)
    print(df[col].unique(), list(range(df[col].nunique())))

    df[col] = df[col].replace(
        df[col].unique(),
        range(df[col].nunique())
    )

    print('*' * 90)
    print()


In [ ]:
df

Some of the value present inside the columns maybe irrelevant so we will try to replace those with appropriate ones

In [ ]:
df['Cholesterol'].value_counts()

There is some mistakes in calculation as there is no chance that someone has 0 cholesterol

In [ ]:
# Mean Median Mode Are Applicable but we will Try applying the KNN Method for now


In [ ]:
df['Cholesterol'].replace(0,np.nan,inplace = True)

In [ ]:
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=3)
after_impute = imputer.fit_transform(df)
df = pd.DataFrame(after_impute,columns=df.columns)

In [ ]:
# Lets check the changes
df['Cholesterol'].isnull().sum()

In [ ]:
# We also can do this
count = 0;
for i in df['Cholesterol']:
  if i==0:
    count+=1
print(count)

In [ ]:
#Checking where the value of RestingBp is 0
df['RestingBP'][df['RestingBP']==0]

In [ ]:
from sklearn.impute import  KNNImputer
df['RestingBP'].replace(0,np.nan)
imputer = KNNImputer(n_neighbors=3)
after_impute = imputer.fit_transform(df) # Numpy array
df = pd.DataFrame(after_impute,columns = df.columns) # Conversion from numpy array to Pandas DataFrame


In [ ]:
df['RestingBP'].isnull().sum()

In [ ]:
df['RestingBP'].unique()

In [ ]:
df.info()

# We can see that all the columns are having  float as their datatype so we will chnage to int except for the Oldpeak because it contains values that will effect the overall process if changed into int leading to errors Check the below code for the type of data present in the oldpeak column

In [ ]:
df['Oldpeak'].unique()

In [ ]:
cols_except_oldpeak = df.columns.drop('Oldpeak')
df[cols_except_oldpeak] = df[cols_except_oldpeak].astype('int32')


In [ ]:
df.corr()

Data Visualizations

In [ ]:
!pip install plotly

In [ ]:
import plotly.express as px

In [ ]:
# Under evaluation we can see tha comparing HeartDisease and target Gives us 1 So we shouldnot compare them, so removing the last one .df.corr()['HeartDisease']
df.corr()['HeartDisease'][:-1]

In [ ]:
px.line(df.corr()['HeartDisease'][:-1].sort_values())

In [ ]:
px.sunburst(df,path=['HeartDisease','Age'])

In [ ]:
px.histogram(df,x='Age',color='HeartDisease')

In [ ]:
px.pie(df,names = 'HeartDisease',title='Percentages of HeartDisease Distributions')

In [ ]:
px.histogram(df,x='Sex',color='HeartDisease')

In [ ]:
px.histogram(df,x='ChestPainType',color='HeartDisease')

In [ ]:
df['RestingBP'].unique()

In [ ]:
px.sunburst(df,path=['HeartDisease','RestingBP'])

In [ ]:
px.histogram(
    df,
    x='FastingBS',
    color='HeartDisease',
    barmode='group'
)


In [ ]:
px.violin(
    df,
    x='HeartDisease',
    y='Oldpeak',
    box=True,
    points='all',
    title='Oldpeak distribution by Heart Disease'
)


In [ ]:
px.violin(df,x='HeartDisease',y='MaxHR',color='HeartDisease')

In [ ]:
px.scatter(
    df,
    x='MaxHR',
    y='Oldpeak',
    color='HeartDisease',
    title='MaxHR vs Oldpeak'
)


In [ ]:
px.histogram(
    df,
    x='ST_Slope',
    color='HeartDisease',
    barmode='group',
    title='ST_Slope vs Heart Disease'
)


In [ ]:
px.histogram(df,x='ExerciseAngina',color ='HeartDisease')

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop('HeartDisease',axis=1),df['HeartDisease'],test_size = 0.2,random_state = 42,stratify = df['HeartDisease']
                                                 )


Heart Disease like is the column that needs to be predicted so we are removing it using drop ,test_size = 0.2 means 20% of the dataset for testing purpose,stratify means that out of all example if df['HeartDisease'] 1 = 60% and 0 = 40% then this thing will be equally splitted in the training and testing dataset while performing actions

In [ ]:
# Binary Classification is Present so we are going to go with Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
solvers = ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']

scores = {}

for s in solvers:
    lr = LogisticRegression(
        solver=s,
        max_iter=2000
    )
    lr.fit(X_train_scaled, y_train)
    scores[s] = lr.score(X_test_scaled, y_test)

In [ ]:
best_solver = max(scores, key=scores.get)
print("Best Solver:", best_solver)
print("Best Accuracy:", scores[best_solver])

In [ ]:
final_lr = LogisticRegression(
    solver=best_solver,
    max_iter=2000
)

final_lr.fit(X_train_scaled, y_train)
lr_pred = final_lr.predict(X_test_scaled)

print("Final Logistic Regression Accuracy:",
      accuracy_score(y_test, lr_pred))

In [ ]:
from sklearn.pipeline import Pipeline
import joblib

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(solver=best_solver, max_iter=2000))
])

pipeline.fit(X_train, y_train)
joblib.dump(pipeline, "model.pkl")

# Support Vector Machine


SVM is a distance based Algorithm so scaling is must otherwise i can create wrong boundaries ,Here we wont be doing fit in test as there is chance of data leakage .We are using SVM because like the data is linearly not separable or due to boundary complex, so svm separates with best possible boundary.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import f1_score

kernels = ['linear', 'poly', 'rbf', 'sigmoid']
scores = {}

for k in kernels:
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel=k))
    ])

    pipeline.fit(X_train, y_train)
    yhat = pipeline.predict(X_test)
    scores[k] = f1_score(y_test, yhat, average='weighted')

best_kernel = max(scores, key=scores.get)
print("Best kernel:", best_kernel)

In [ ]:
import joblib

final_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel=best_kernel))
])

final_svm.fit(X_train, y_train)

joblib.dump(final_svm, "svm_model.pkl")

f1_score is used because for the imbalanced data it is the best . Standard Scaler is used beacuse it is the best and compulsory for SVM .Scaling is bringing every element within the same range.Use of  kernels in this code is telling how the data must be separated and it is applying projection in higher dimension so separation is possible  kernels like linear is used if data is linearly separable and poly for polynimial relation is there .rbf means Radial Basis Function, Full name Gaussian RBF Kernel it projects data in infinite dimensional space . Sigmoid is inspired from neural networks uses weird S shape and rarely used .Majorly rbf wins and it can be any shape.

Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score
import joblib

dtree = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42
)

param_grid = {
    'max_depth': [3, 5, 7, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    dtree,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_tree = grid_search.best_estimator_

dtc_pred = best_tree.predict(X_test)

print("Best Params:", grid_search.best_params_)
print("DecisionTree F1 Score:", f1_score(y_test, dtc_pred))

In [ ]:
joblib.dump(best_tree, "decision_tree_model.pkl")

we use GridSearchCv because it is a hyperparameter tuning tool so model becomes balanced.why using class_weight as balanced beacuse in the  HeartDisease dataset is imbalanced there is more 0 and less 1 if balanced isnot used model will tell everyone 0 increasing the accuracy, Here is HeartDisease Column with 1 is very important though it falls under minority. Things like max depth ,min samples split ,min sample leaf and random state are just the hyperparameters .Here cv = 5 is saying do 5 fold cross validation means divide data into 5 parts 4 train and test in 1 and get the average for generalization purpose.** means unpacking

In [ ]:
# RandomForestClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score
import joblib

rfc = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [6, None],
    'max_features': ['sqrt'],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    rfc,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

rfc_best = grid_search.best_estimator_

pred_rfc = rfc_best.predict(X_test)

print("Best Params:", grid_search.best_params_)
print("RandomForest F1 Score:", f1_score(y_test, pred_rfc))

In [ ]:
joblib.dump(rfc_best, "random_forest_model.pkl")

Using randomForest because it is like alot of  decison trees with little randomness

MODEL TRAINING